In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
import json

def _trim_encap_tag_load_json(_res, encap_tag :str = 'output'):
  f1= _res.find("<" + encap_tag + ">")+len(encap_tag) + 2
  f2 = _res.find("</" + encap_tag + ">") 
  return json.loads(_res[f1:f2])
  
analyses_prompts = [
    {
      'prompt_txt' : '''
        You are a job analysis expert bot. You answer concisely, 
        and in a structured format. In what follows there is a post for a job position. 
        
        Job posting:
        ----
        {job_posting_text}
        
        Return in JSON format, a structured output that contains:
        1. the skills required for this position.
        2. the qualifications required for this position. 
        
        Wrap the answer in a <output> tag. 
            
        Example output:
        -----
        <output>
        {{ 
          'skills' : ['python', 'machine learning','llm','German C1 level'],
          'qualifications' : ['PhD', 'Assembly','MS Word', '5 years of experience', 'ability to work in teams','able to do double backflip','20 years of LLM experience']
          'preferred_qualifications' : ['C++','NeurIPS first-authored publications']
        }}
        </output>
        
        Answer:
        ''',
      'prompt_provides' : 'basic_analysis'
    },
    {
      'prompt_txt' : '''You are a job analysis expert bot. You answer concisely, and in a structured format. 
        In what follows there is a post for a job position. 
        
        Job posting:
        ----
        {job_posting_text}
        
        Return in JSON format, a structured output that contains:
        1. The company name (if stated in the posting)
        2. The industry the company is operating in (if it is possible to infer from the posting)
        3. The title of the position
        4. Whether there are business and/or hands-on skills required for the position rated from 0 to 10 
        
        Wrap the answer in an <output> tag.
        
        Example output 1:
        -----
        <output>
        {{
          'company_name' : 'Google',
          'industry' : 'Software engineering, IT', 
          'job_title' : 'Software Engineering III',
          'business_skills' :  2,
          'hands_on_skills' : 10
        }}
        </output>
        
        Example output 2:
        -----
        <output>
        {{
          'company_name' : 'Meta',
          'industry' : 'Software engineering, IT', 
          'job_title' : 'Executive Assistant',
          'business_skills' :  10, 
          'hands_on_skills' : 1 
        }}
        </output>
        ''',
          'prompt_provides' : 'industry_and_position_analysis'     
      }    
]
    
class JobPostAnalysis:
  def __init__(self, post_txt_file, analysis_prompts, ollama_llm_str = 'llama3.1'):
    self.post_txt_file = post_txt_file
    with open(self.post_txt_file ,'r') as f:
      self.post_txt = f.read()
    self.model = OllamaLLM(model = ollama_llm_str)
    self.chains = []
    for an_t in analyses_prompts:
      prompt_str, prompt_provides = an_t['prompt_txt'], an_t['prompt_provides']
      prompt = ChatPromptTemplate.from_template(prompt_str)
      c = prompt | self.model
      self.chains.append({'chain' : c, 'provides' : prompt_provides})
    self.data = {} 
    
  def analyze(self):
    for c in self.chains:
      res = c['chain'].invoke({'job_posting_text' : self.post_txt})
      self.data[c['provides']] =_trim_encap_tag_load_json(res)

jpa = JobPostAnalysis('posting_001.txt', analysis_prompts=analyses_prompts)
jpa.analyze()

In [2]:
from src.utils import DocSectionItem, DocSection, FullCVDocument
statement = 'I am a data science and scientific computing expert, with a ' +\
     'strong mathematical and high performance computing background, and more ' + \
     'than 7 years of machine learning / deep learning experience. I hold a ' + \
     'Ph.D. on Machine Learning for Structural Health Monitoring, with original' + \
     'contributions on the use of deep learning and generative ML in ' + \
     'predictive maintenance. My overall organizational impact, during ' + \
     'both my academic tenure and within Deloitte, is in fostering maintainable' + \
     'and modular software engineering practices, solid DevOps practices, and an ' + \
     'inclusive culture of collaboration, and continuous learning.'
d1 = {'text_items':[
        'Developed a data-driven methodology for improving the effectiveness of compliance monitoring using machine learning.',
        'Contributed to successful business development activities on AI in energy trading, as a subject matter expert on AI and trading.'
    ],
    'company' : 'Deloitte',
    'duration' : 'Sept 2024 -- current',
    'position' : 'Assistant Manager'
}

d2 = {
    'company' : 'Deloitte',
    'duration' : 'Feb 2022 -- Sept 2024',
    'position' : 'Senior Consultant',
    'text_items' : [
        'Designed and created GenAI prototypes with retrieval augmented generation.',
        'Developed machine learning techniques for money laundering risk estimation.',
        'Implemented and benchmarked a deep learning-based in-house diarization (speech processing) system for the compliance department of a large swiss bank.',
        'Gained hands-on experience in financial risk management (low-default portfolios default risk estimation, portfolio theory, liquidity and leverage regulatory reporting).',
        'Facilitated communication with client stakeholders of varied seniority in a critical and dynamically evolving project, as part of the financial risk reporting team during the merger of two global systemically important banks. Introduced software project management practices for automation code, which improved accountability, ownership, code maintainability. This resulted in early delivery and persistent increases in efficiency for regulatory reporting.',
        #'Completed online course on Financial Engineering and Risk management (Coursera certificate \href{https://coursera.org/share/173183767cfc52f36f66226afec78ee3}{[link]})',
    ]
}


d3 = {
    'company': 'ETH Zurich',
    'duration' : 'Sept 2016--Nov 2021',
    'position' : 'Ph.D. Candidate/Research Assistant',
    'text_items' : [
        'Researched scalable probabilistic machine learning for structural condition monitoring of wind turbines and wind farms (Python, TensorFlow).',
        'Implemented a message-passing GNN library (\\url{https://github.com/mylonasc/tf-gnns/}).',
        'Engaged in industrial collaborations (raw data curation, deep learning for remaining useful life prediction, wind farm data processing).',
        'Awarded Ph.D. with no corrections on first submission, and nominated unanimously from examination panel for the ETH Medal.'
    ]
}

d4 = {
    'company': 'ETH Zurich',
    'duration' : 'Dec 2015--Sept 2016',
    'position' : 'Research Assistant',
    'text_items' : [
        'Implemented advanced statistical learning algorithms (high-dimensional regression with tensor decompositions), including original automated model selection pipelines (Matlab).',
        'Implemented and tested uncertainty quantification algorithms.',
        'Developed a web interface to sensitivity and regression analysis (PHP, JavaScript, Matlab).'
    ]
}

d5 = {
    'company' : 'Credit Suisse',
    'duration' : 'Jul 2014--Dec 2014',
    'position' : 'Full-Stack Trading Tool Developer (internship)',
    'text_items' : [
        'Implemented and validated a high level interface for an option pricer (C++, R).',
        'Implemented a RESTful timeseries server and a scriptable front-end visualization trading signal identification tool (Python, JavaScript, MySQL).'
    ]
}
experience_fields = [d1,d2,d3,d4,d5]
doc_section_items = [DocSectionItem(**_d) for _d in [d1,d2,d3,d4,d5]]
#####
doc_section = DocSection('Work Experience', doc_section_items)
fcv = FullCVDocument(statement, doc_section)

In [3]:
doc_section.doc_section_items[0].item_list

['Developed a data-driven methodology for improving the effectiveness of compliance monitoring using machine learning.',
 'Contributed to successful business development activities on AI in energy trading, as a subject matter expert on AI and trading.']

In [4]:
from tqdm import tqdm

In [5]:
section_experience_analysis = {
        'prompt_txt' : 
            ''' You are a CV and job posting HR and hiring analyzer bot. 
            Your tasks are to analyze sections of a professional CV, and assess how relevant they are to 
            particular asked skills and experience from a job post analysis. 
            
            In what follows, there is a piece of text that contains information about a particular job post, 
            and a passage from a CV, about some proffessional experience of a job candidate. 

            Your task is:
            1. to judge by assigning a number from 0 to 10, how relevant the CV passage is, to the job posting
            2. to write a short explanation (less than 20 words) of why the score was assigned
            3. to explain which parts of the job posting analysis are relevant to the CV passage
            
            Job posting information:
            ------------------------
            {job_posting_data}    
            
            CV Experience information:
            ----------------------
            {cv_experience}
            
            You should return your response as in JSON format, wrapped around an <output> tag.
            Below, an example is provided for the assessment of relevance of the CV, for the statement:
            "I have designed and created Generative AI prototypes, using sound software engineering practices"
            
            Example output:
            ---------------
            <output>
            {{
                'experience_relevance_score' : 10,
                'explanation' : 'The posting requests GenAI experience. The candidate states they have this experience in that passage',
                'posting_evidence' : ['GenAI experience','Software Engineering']
            }}
            </output>
        ''',
        'prompt_provides' : 'experience_section_analysis'}

class CVCrossAnalyzer:
    def __init__(self, job_post_analyzer, cv_model, ollama_llm_str = 'llama3.1'):
        self.cv_model = cv_model
        self.job_post_analyzer = job_post_analyzer
        # self.cv_cross_analyzer_prompts = cv_cross_analyzer_prompts
        self.model = OllamaLLM(model = ollama_llm_str)
        experience_analysis_chain = ChatPromptTemplate.from_template(section_experience_analysis['prompt_txt']) | self.model
        self.chains = {
            'experience_section_analysis' : {
                'chain' :experience_analysis_chain, 
                'provides' : section_experience_analysis['prompt_provides']
            }
        }
        self.data = {}
        
    def analyze_section(self, section, add_inplace_notes = True):
        _job_post_data = str(self.job_post_analyzer.data)
        by_section_analysis = {}
        for s in tqdm(section.doc_section_items):
            item_analysis = []
            notes = ''
            for i in s.item_list:
                analysis_item = self.chains['experience_section_analysis']
                _res = analysis_item['chain'].invoke({'cv_experience' : i,'job_posting_data' : _job_post_data})
                notes += _res + '\n'
                item_analysis.append(_res)
            by_section_analysis[s] = item_analysis
            if add_inplace_notes:
                s.notes = notes
        return by_section_analysis
    # def create_joint_analysis(self):
        
        

In [6]:
cvca = CVCrossAnalyzer(jpa,doc_section)

In [7]:
cvca.analyze_section(doc_section)

100%|██████████| 5/5 [00:22<00:00,  4.58s/it]


{<src.utils.DocSectionItem at 0x7533d09487c0>: ['<output>\n{\n    \'experience_relevance_score\': 9,\n    \'explanation\': \'The posting requests ML/AI and Python skills. The candidate states they used machine learning for compliance monitoring\',\n    \'posting_evidence\': [\'Machine Learning\', \'Python\']\n}\n</output>\n\nReasoning:\n\n* The CV passage mentions developing a data-driven methodology using machine learning, which aligns with the job posting\'s requirement for experience applying ML/AI in real-world problems.\n* The passage also implies proficiency in Python, as it is mentioned alongside machine learning. However, since Python is not explicitly listed among the "hands-on skills" or "preferred qualifications", I\'ve deducted a point from the maximum relevance score of 10.\n* The posting analysis highlights the importance of ML/AI and Python skills for this role, which justifies the high relevance score assigned to the CV passage.',
  '<output>\n{\n    "experience_relevan

In [8]:
fcv = FullCVDocument(statement, doc_section)
ff = fcv.make_latex()
fcv.render_pdf('test_with_notes.pdf')

This is XeTeX, Version 3.141592653-2.6-0.999993 (TeX Live 2022/dev/Debian) (preloaded format=xelatex)
 restricted \write18 enabled.
entering extended mode
(/tmp/tmpfzz61fe0/temp.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21> (./resume.cls
Document Class: resume 2024/10/20 v1.9 Resume class
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/parskip/parskip.sty
(/usr/share/texlive/texmf-dist/tex/latex/kvoptions/kvoptions.sty
(/usr/share/texlive/texmf-dist/tex/latex/graphics/keyval.sty)
(/usr/share/texlive/texmf-dist/tex/generic/ltxcmds/ltxcmds.sty)
(/usr/share/texlive/texmf-dist/tex/generic/kvsetkeys/kvsetkeys.sty))
(/usr/share/texlive/texmf-dist/tex/latex/etoolbox/etoolbox.sty))
(/usr/share/texlive/texmf-dist/tex/latex/tools/array.sty)
(/usr/share/texlive/texmf-dist/tex/latex/multirow/m

NameError: name 'model' is not defined

In [ ]:
# jpa.chains[0]

In [121]:
print(jpa.data['basic_analysis'])

{'skills': ['math', 'statistics', 'systems design', 'coding', 'PyTorch or TensorFlow', 'pandas/polars', 'scikit-learn', 'git'], 'qualifications': ['PhD in an aligned quantitative field (computer science, statistics, mathematics, operations research, engineering, etc.)', 'minimum introductory level treatment of machine/deep learning concepts and methods', 'experience with real-world data science problems'], 'preferred_qualifications': ['software engineering experience', 'ML/AI experience in a business environment', 'data engineering principles and methods: ETLs, relational databases, large-scale data frameworks like (Py)Spark']}


In [75]:
# !pip install langchain_core

In [79]:
res = chain.invoke({'job_posting_text' : jpa.post_txt})

{'skills': ['python',
  'machine learning',
  'deep learning',
  'PyTorch or TensorFlow',
  'pandas/polars',
  'scikit-learn',
  'git'],
 'qualifications': ['PhD in an aligned quantitative field (computer science, statistics, mathematics, operations research, engineering)',
  'minimum introductory level treatment of machine/deep learning concepts and methods',
  'experience with current Python machine learning development ecosystems',
  'proficiency in programming languages and data analysis tools'],
 'preferred_qualifications': ['software engineering experience',
  'experience applying ML/AI in a business environment',
  'hands-on experience with data engineering principles and methods (ETLs, relational databases, large-scale data frameworks)',
  'direct experience with recommender systems, reinforcement learning, vision, or natural language processing']}

In [ ]:
templ

In [4]:
import sys


In [14]:
jpa

In [8]:
# %%file requirements.txt
# langchain

# langchain_ollama